# Opis

Krótki notebook, który pozwala przetestować działanie różnych elementów implementacyjnych w szybki sposób.

# Importy

In [1]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import tqdm
import wandb
import json
sys.path.append('../') # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.GraphAutoencoder import GraphAutoencoder
from src.models.KlejdaGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset

C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import sys
current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)
import frams

frams.init(
    evolution_config['frams_path']
)

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data



# Przetwarzanie

## Załadowanie danych i przygotowanie do przetwarzania

In [3]:
# dataset = FramsticksDummyDataset(num_samples=1000)
torch.set_float32_matmul_precision('high')
genotypes = []
with open("../results/sampled_best_individuals_merged.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        genotypes.append(obj)

with open("../configs/klejda_gae_config.yaml") as f:
    config = yaml.safe_load(f)

dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])

dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(
    train_dataset,
	# TODO: ile ustawić? Może też powinno być w configs, tak jak wszystko inne?
    batch_size=256,
    shuffle=True,
    num_workers=4,
    persistent_workers=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=4,
    persistent_workers=True
)

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\witek\_netrc.
wandb: Currently logged in as: witekadrian7 (witekadrian7-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## GAE

In [4]:
modelGAE = GraphAutoencoder(config, frams_module=frams).double()

In [7]:
wandb_logger = WandbLogger(project="Framsticks-GAE", name="GAE-Baseline-Test", save_dir = config["save_dir"])

trainer = pl.Trainer(
    max_epochs=160,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)
trainer.fit(modelGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: setting up run swfl6pbd
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260819_195617-swfl6pbd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run GAE-Baseline-Test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-GAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-GAE/runs/swfl6pbd
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │ 70.9 K │ train │     0 │
│ 1 │ fc_z             │ Linear   │    650 │ train │     0 │
│ 2 │ decoder_a        │ DecoderA │ 38.4 K │ train │     0 │
│ 3 │ decoder_x        │ DecoderX │ 36.4 K │ train │     0 │
│ 4 │ criterion        │ MSELoss  │      0 │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 146 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 146 K                                                                                                
Total estimated model params size (MB): 0.586                                                                      
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

`Trainer.fit` stopped: `max_epochs=160` reached.


wandb: updating run metadata
wandb: uploading config.yaml; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█
wandb: train/locality_correlation ▇▇██████▇█▇███▇▇▇▇▇▇▃▇▇▇▆▆▇▇▆▆▅▄▅▅▅▅▄▅▁▄
wandb:               train/loss_A ▅▂▂▂▁▁▁▁▂▂▂▂▂▃▃▂▃▂▄▃▃▃▃▂▂▃▃▃▄▄▅▅▆▆▅▆▆▆██
wandb:               train/loss_X ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▂
wandb:        train/loss_locality ▆▂▂▂▁▁▁▁▁▂▂▂▂▂▂▂▃▂▃▃▄▃▃▄▃▄▄▅▄▄▆▅▅▆█▆▇▇▇▇
wandb:           train/loss_total ▂▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▂▃▄▃▃▃▃▃▃▄▄▄▄▅▅▆▅▅▅█▇███
wandb:        trainer/global_step ▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇██████
wandb:   val/locality_correlation ▇▇▇▇███▇███▇▇▇█▇▇▇▆▆▇▆▆▅▆▇▆▇▆▆▅▅▆▆▆▆▆▁▄▅
wandb:                 val/loss_A ▂▁▁▁▁▁▁▁▁▂▁▂▂▂▂▂▂▃▃▃▃▃▄▃▂▄▄▃▃▃▄▃▅▄▅▇▅▅▆█
wandb:                 val/loss_X ▂▁▁▁▁▁▁▂▃▂▂▂▂▂▃▃▄▃▃▃▃▄█▄▃▄▄▄▄▅▄▅▅▇▅▆▆▅▇▆
wandb:                         +2 ...
wandb: 
wandb: Run summary:
wandb:                      epoch 159
wandb: train/locality_correlat

## VGAE

In [4]:
modelVGAE = VariationalGraphAutoencoder(config, frams_module=frams).double()


In [7]:

wandb_logger = WandbLogger(project="Framsticks-VGAE", name="VGAE-Baseline-Test", save_dir = config["save_dir"])
trainer = pl.Trainer(
    max_epochs=160,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)

trainer.fit(modelVGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: setting up run k9k9s86r
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260816_210233-k9k9s86r
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run VGAE-Baseline-Test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-VGAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-VGAE/runs/k9k9s86r
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │ 70.9 K │ train │     0 │
│ 1 │ fc_mu            │ Linear   │  6.5 K │ train │     0 │
│ 2 │ fc_logvar        │ Linear   │  6.5 K │ train │     0 │
│ 3 │ decoder_a        │ DecoderA │ 44.2 K │ train │     0 │
│ 4 │ decoder_x        │ DecoderX │  189 K │ train │     0 │
│ 5 │ criterion        │ MSELoss  │      0 │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 317 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 317 K                                                                                                
Total estimated model params size (MB): 1.269                                                                      
Modules in train mode: 70                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=160` reached.


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 314-319, summary
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▁▁▁▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇███
wandb: train/locality_correlation ▂▄▂▁▃▃▃▂▃▄▃▄▂▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▄▆▇▇▇▇▇████
wandb:               train/loss_A ▅▅▅▆█▅▄▅▅▄▄▄▃▃▃▃▃▂▃▂▂▂▂▂▁▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁
wandb:              train/loss_KL ████▆▆▆█▅▅▅▄▇▅▅▄▄▅▄▃▂▃▃▅▃▃▂▄▂▄▂▂▂▁▂▁▁▂▁▁
wandb:               train/loss_X █▃▄▅▅▃▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁
wandb:        train/loss_locality ▅▅▄▆▆▄▅▄▄█▅▄▄▄▄▃▃▃▂▃▂▂▂▂▂▂▂▄▂▂▂▃▂▁▁▁▁▁▂▂
wandb:           train/loss_total █▅▅▅▅▅▄▆▆▅▄▃▄▄▃▂▃▂▂▂▂▂▃▂▂▂▂▁▂▂▂▂▁▂▂▁▁▁▁▁
wandb:        trainer/global_step ▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█
wandb:   val/locality_correlation ▄▂▄▃▁▃▄▁▂▄▄▄▄▄▅▇▆▆▆▆▆▇▆▆▇▇▇▇▇▇▇▇████████
wandb:                 val/loss_A ▆▅▅▄▆▅▄▄█▃▂▃▂▃▃▃▃▃▂▂▂▁▁▂▂▁▂▂▂▂▂▂▁▂▁▁▁▂▂▁
wandb:                         +4 ...
wandb: 
wandb: Run summary:
wandb:              